In [ ]:
# =========================
# Colab Setup
# =========================
import sys
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

import subprocess
# spikeinterface and ipympl from PyPI
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "spikeinterface[extractors]", "ipympl", "pyyaml"])
# braindance from a local copy on Drive (no PyPI release)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                       "/content/drive/MyDrive/Kosik-Lab/packages/braindance"])

# Make the casi package importable from Drive
_CASI_ROOT = Path("/content/drive/MyDrive/Kosik-Lab/CASI")
sys.path.insert(0, str(_CASI_ROOT / "src"))


In [ ]:
%matplotlib inline
import gc
from casi.config import load_configs, render_summary
from casi.engine import (
    run_pipeline,
    create_registry,
    get_recording_duration_ms,
    clear_data_caches,
)


# Configuration

In [ ]:
# Load configs from the CASI repo on Drive, then override the paths for Colab.
cfg = load_configs(_CASI_ROOT / "configs")

_DRIVE = "/content/drive/MyDrive/Kosik-Lab"
cfg.paths.update({
    "mea_h5":               f"{_DRIVE}/data/alignment-data/Test-B/MEA_B.raw.h5",
    "ca_traces_npz":        f"{_DRIVE}/data/alignment-data/Test-B/CA_traces_B.npz",
    "rt_model_outputs_npy": f"{_DRIVE}/data/alignment-data/Test-B/sequences/model_outputs.npy",
    "rt_model_path":        f"{_DRIVE}/models/rtsort_model",
    "output_dir":           "/content/casi_outputs",
    "cache_dir":            "/content/casi_cache",
})
# Colab-specific globals overrides
cfg.globals.update({
    "plot_backend": "matplotlib",
    "interactive_plots": False,
    "memory_cache": False,
    "disk_cache": False,
})

paths_cfg  = cfg.paths
ops_cfg    = cfg.ops
globals_cfg = cfg.globals

print(render_summary(cfg))


In [ ]:
pipe = [
    "window_ms[14487.05, 44352.95]",
    # "window_ms[full]",
    "mea_trace(861)",
    "mea_trace(861).saturation_mask",
    "mea_trace(861).saturation_mask.notch_filter",
    "mea_trace(861).saturation_mask.notch_filter.butter_bandpass",
    "mea_trace(861).saturation_mask.notch_filter.butter_bandpass.amp_gain_correction",
    "mea_trace(861).saturation_mask.notch_filter.butter_bandpass.amp_gain_correction.rt_detect",
    "mea_trace(861).saturation_mask.notch_filter.butter_bandpass.amp_gain_correction.rt_detect.sigmoid",
    "mea_trace(861).saturation_mask.notch_filter.amp_gain_correction.rt_detect.sigmoid.rt_thresh",
    # "mea_trace(861).saturation_mask().notch_filter.rt_detect",
    # "mea_trace(861).freq_traces",
    # "mea_trace(861).saturation_mask.freq_traces",
]


rt_window_pipe = [
    "window_ms[14487.05, 44352.95]",
    # "window_ms[full]",
    "mea_trace(861)",
    "mea_trace(861).notch_filter",
    "mea_trace(861).rt_detect",
    "mea_trace(861).notch_filter.rt_detect",
    "rtsort(861)",
    "mea_trace(861).rt_detect.sigmoid",
    "rtsort(861).sigmoid",
    "mea_trace(861).rt_detect.sigmoid.rt_thresh",
    "rtsort(861).sigmoid.rt_thresh",
]


bank = [
    "window_ms[15000, 16000]",
    "mea_trace(861).butter_bandpass.sliding_rms",
    ["mea_trace(861)", "mea_trace(861).butter_bandpass"],
    "mea_trace(861).butter_bandpass.spectrogram",
    "mea_trace(861).freq_traces",
    "ca_trace(11).baseline_correction.x_corr(mea_trace(all).butter_bandpass.sliding_rms.gcamp_sim)"
    "mea_trace(861).butter_bandpass.spectrogram",
]

pipeline_cfg = pipe

# Run Pipeline

In [ ]:
plot_backend = globals_cfg.get("plot_backend", "matplotlib")
registry, source_loaders = create_registry(plot_backend=plot_backend)

results = run_pipeline(
    paths_cfg=paths_cfg,
    ops_cfg=ops_cfg,
    pipeline_cfg=pipeline_cfg,
    registry=registry,
    source_loaders=source_loaders,
    globals_cfg=globals_cfg,
    mea_fs_hz=20000.0,
    rtsort_fs_hz=20000.0,
    verbose=True,
    plot=True,
    get_recording_duration_ms=get_recording_duration_ms,
)

clear_data_caches()
gc.collect()
